# 00 — Setup, download, raw AnnData, and SpatialData (interactive)

This notebook replaces `%run` with inspectable cells. Run one section at a time. Expensive write operations are controlled by explicit `WRITE_*` switches.

In [69]:
# ============================================================
# Dataset_02 CosMx — Jupyter initialization
# ============================================================

from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc


# ------------------------------------------------------------
# Robustly locate project root
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """
    Search current directory and its parents for the project root.

    A valid project root contains:
        src/
        data/
    """

    for candidate in [start, *start.parents]:

        if (
            (candidate / "src").is_dir()
            and
            (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root.\n"
        f"Starting directory: {start}\n\n"
        "Expected a parent directory containing:\n"
        "  src/\n"
        "  data/"
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)


# ------------------------------------------------------------
# Make src importable
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ------------------------------------------------------------
# Common paths
# ------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

SPATIAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "spatial"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

CONFIG_DIR = (
    PROJECT_ROOT
    / "config"
)


# ------------------------------------------------------------
# Scanpy settings
# ------------------------------------------------------------

sc.settings.verbosity = 2


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("Dataset_02 CosMx Jupyter environment")
print("=" * 70)

print("Notebook directory :", CURRENT_DIR)
print("Project root       :", PROJECT_ROOT)
print("Python             :", sys.executable)

print()

print(
    "src exists         :",
    (PROJECT_ROOT / "src").exists(),
)

print(
    "raw data exists    :",
    RAW_DIR.exists(),
)

print(
    "processed exists   :",
    PROCESSED_DIR.exists(),
)

print(
    "spatial exists     :",
    SPATIAL_DIR.exists(),
)

Dataset_02 CosMx Jupyter environment
Notebook directory : /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/jupyter-2
Project root       : /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised
Python             : /home/jqu/.conda/envs/spatialdata/bin/python

src exists         : True
raw data exists    : True
processed exists   : True
spatial exists     : True


In [70]:
import spatialdata, dask, pandas, zarr, anndata
import spatialdata_plot
print("spatialdata     :", spatialdata.__version__)
print("spatialdata-plot:", spatialdata_plot.__version__)
print("scanpy          :", sc.__version__)
print("anndata         :", anndata.__version__)
print("dask            :", dask.__version__)
print("pandas          :", pandas.__version__)
print("zarr            :", zarr.__version__)

spatialdata     : 0.7.2
spatialdata-plot: 0.3.3
scanpy          : 1.11.5
anndata         : 0.12.11
dask            : 2026.1.1
pandas          : 2.3.3
zarr            : 3.1.6


/lsf_tmp/323066199.tmpdir/ipykernel_1573392/2800429130.py:5: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy          :", sc.__version__)
/lsf_tmp/323066199.tmpdir/ipykernel_1573392/2800429130.py:6: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata         :", anndata.__version__)


## A. Programmatic GEO download
Set `DOWNLOAD=True` only when you want to download/check the five GEO files.

In [71]:
from urllib.request import urlretrieve
from src.cosmx_io import FILES, BASE_URL, validate_gzip_csv

RAW_DIR = PROJECT_ROOT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD = False

if DOWNLOAD:
    for label, filename in FILES.items():
        path = RAW_DIR / filename
        if path.exists():
            try:
                validate_gzip_csv(path)
                print("OK/skip:", label, path.name, f"({path.stat().st_size/1e6:.1f} MB)")
                continue
            except Exception as e:
                print("Existing file invalid; redownloading:", path, e)
        url = BASE_URL + filename
        print("Downloading", url)
        urlretrieve(url, path)
        validate_gzip_csv(path)
        print("Saved:", path)
else:
    print("DOWNLOAD=False; existing raw files:")
    for label, filename in FILES.items():
        p = RAW_DIR / filename
        print(f"  {label:12s}", p.exists(), p)

DOWNLOAD=False; existing raw files:
  expression   True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_exprMat_file.csv.gz
  metadata     True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_metadata_file.csv.gz
  fov          True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_fov_positions_file.csv.gz
  polygons     True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_polygons.csv.gz
  transcripts  True /research/rgs01/home/clus

## B. Build and inspect raw AnnData
This calls the reusable `src.cosmx_io.build_raw_anndata()` function directly, not a production script.

In [72]:
from src.cosmx_io import build_raw_anndata

ADATA_PATH = PROJECT_ROOT / "data/processed/GSM9046088_CosMx_raw.h5ad"
WRITE_ANNDATA = False

adata_raw = build_raw_anndata(RAW_DIR)
print(adata_raw)
display(adata_raw.obs.head())
print("spatial:", adata_raw.obsm.get("spatial", np.empty((0,2))).shape)
print("counts layer:", "counts" in adata_raw.layers)

#if WRITE_ANNDATA:
#    ADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
#    adata_raw.write_h5ad(ADATA_PATH, compression="gzip")
#    print("Saved:", ADATA_PATH)

AnnData object with n_obs × n_vars = 43565 × 1010
    obs: 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.B2M.MembraneStain', 'Max.B2M.MembraneStain', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI'
    uns: 'spatial_coordinate_columns', 'fov_positions', 'GEO_accession', 'platform', 'sample'
    obsm: 'spatial'
    layers: 'counts'


,fov,cell_ID,Area,AspectRatio,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px,Width,Height,Mean.PanCK,Max.PanCK,Mean.CD68,Max.CD68,Mean.B2M.MembraneStain,Max.B2M.MembraneStain,Mean.CD45,Max.CD45,Mean.DAPI,Max.DAPI
unique_cell_id,,,,,,,,,,,,,,,,,,,,
fov_1_cell_1,1,1,7538,0.95,2515,4056,-495476.6667,11339.33333,99,104,1183,4581,0,105,108,458,2,128,1,87
fov_1_cell_2,1,2,4969,1.11,3831,3925,-494160.6667,11208.33333,83,75,143,1746,0,93,53,787,3,530,34,306
fov_1_cell_3,1,3,4045,0.75,2368,3901,-495623.6667,11184.33333,65,87,31,378,5,829,344,1389,18,1284,105,397
fov_1_cell_4,1,4,7853,0.86,2921,3695,-495070.6667,10978.33333,96,111,1041,8586,18,245,172,843,29,589,136,550
fov_1_cell_5,1,5,7992,0.81,3907,3665,-494084.6667,10948.33333,90,111,889,7437,6,310,96,1015,16,909,0,34


spatial: (43565, 2)
counts layer: True


## C. Build SpatialData interactively
This is the same logic as `02_build_sdata.py`, exposed as cells. It can be memory-intensive because the transcript CSV is large.

In [73]:
import dask.dataframe as dd
import geopandas as gpd
from shapely.geometry import Polygon
from spatialdata import SpatialData
from spatialdata.models import PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity
from src.cosmx_io import unique_cell_id

POLY = RAW_DIR / FILES["polygons"]
TX = RAW_DIR / FILES["transcripts"]

poly = pd.read_csv(POLY)
cell_col = "cell_ID" if "cell_ID" in poly.columns else "cellID"
poly["unique_cell_id"] = unique_cell_id(
    pd.to_numeric(poly["fov"]).astype(int),
    pd.to_numeric(poly[cell_col]).astype(int),
)
print(poly.shape)
display(poly.head())

(1197279, 8)


,Unnamed: 0,fov,cellID,x_local_px,y_local_px,x_global_px,y_global_px,unique_cell_id
0,1,1,1,2515,4107,-495476.666667,11390.333333,fov_1_cell_1
1,2,1,1,2522,4106,-495469.666667,11389.333333,fov_1_cell_1
2,3,1,1,2527,4105,-495464.666667,11388.333333,fov_1_cell_1
3,4,1,1,2536,4102,-495455.666667,11385.333333,fov_1_cell_1
4,5,1,1,2538,4101,-495453.666667,11384.333333,fov_1_cell_1


In [74]:
# Build polygons. This may take a little time.
records = []
invalid_fixed = skipped = 0
for uid, g in poly.groupby("unique_cell_id", sort=False):
    xy = g[["x_global_px", "y_global_px"]].to_numpy(float)
    if len(xy) < 3:
        skipped += 1; continue
    geom = Polygon(xy)
    if not geom.is_valid:
        geom = geom.buffer(0); invalid_fixed += 1
    if geom.is_empty:
        skipped += 1; continue
    records.append((uid, geom))

shapes = gpd.GeoDataFrame(
    {"geometry": [x[1] for x in records]},
    index=pd.Index([x[0] for x in records], name="instance_id"), crs=None,
)
shapes_model = ShapesModel.parse(shapes, transformations={"global": Identity()})
print("valid polygons:", len(shapes), "fixed:", invalid_fixed, "skipped:", skipped)

valid polygons: 43565 fixed: 0 skipped: 0


In [75]:
# ============================================================
# Build SpatialData table from raw AnnData
# and link it to cell segmentation polygons
# ============================================================

from spatialdata.models import TableModel

REGION_NAME = "cell_boundaries"
REGION_KEY = "region"
INSTANCE_KEY = "instance_id"


# ============================================================
# 1. Start from RAW AnnData
# ============================================================

tab_adata = adata_raw.copy()

print("=" * 70)
print("BUILD SPATIALDATA TABLE")
print("=" * 70)

print("Raw AnnData:")
print(
    f"  {tab_adata.n_obs:,} cells × "
    f"{tab_adata.n_vars:,} features"
)

print()

print("Shapes:")
print(
    f"  {len(shapes):,} polygons"
)

BUILD SPATIALDATA TABLE
Raw AnnData:
  43,565 cells × 1,010 features

Shapes:
  43,565 polygons


In [76]:
# ============================================================
# 2. Inspect cell identifiers BEFORE matching
# ============================================================

adata_ids = pd.Index(
    tab_adata.obs_names.astype(str)
)

shape_ids = pd.Index(
    shapes.index.astype(str)
)


print("AnnData IDs:")
display(
    adata_ids[:5]
)

print("\nShape IDs:")
display(
    shape_ids[:5]
)


print("\nUnique AnnData IDs:")
print(adata_ids.is_unique)

print("\nUnique shape IDs:")
print(shape_ids.is_unique)


if not adata_ids.is_unique:
    raise ValueError(
        "AnnData obs_names are not unique."
    )

if not shape_ids.is_unique:
    raise ValueError(
        "Shape instance IDs are not unique."
    )

AnnData IDs:


Index(['fov_1_cell_1', 'fov_1_cell_2', 'fov_1_cell_3', 'fov_1_cell_4',
       'fov_1_cell_5'],
      dtype='object', name='unique_cell_id')


Shape IDs:


Index(['fov_1_cell_1', 'fov_1_cell_2', 'fov_1_cell_3', 'fov_1_cell_4',
       'fov_1_cell_5'],
      dtype='object', name='instance_id')


Unique AnnData IDs:
True

Unique shape IDs:
True


In [77]:
# ============================================================
# 3. Compare AnnData cells with segmentation polygons
# ============================================================

matched_ids = adata_ids.intersection(
    shape_ids,
    sort=False,
)

adata_without_shape = adata_ids.difference(
    shape_ids
)

shape_without_adata = shape_ids.difference(
    adata_ids
)


mapping_summary = pd.DataFrame(
    {
        "category": [
            "AnnData cells",
            "Shape polygons",
            "Matched cells",
            "AnnData without polygon",
            "Polygon without AnnData",
        ],
        "n": [
            len(adata_ids),
            len(shape_ids),
            len(matched_ids),
            len(adata_without_shape),
            len(shape_without_adata),
        ],
    }
)

mapping_summary["fraction_of_anndata"] = (
    mapping_summary["n"]
    / len(adata_ids)
)

display(mapping_summary)

,category,n,fraction_of_anndata
0,AnnData cells,43565,1.0
1,Shape polygons,43565,1.0
2,Matched cells,43565,1.0
3,AnnData without polygon,0,0.0
4,Polygon without AnnData,0,0.0


In [78]:
# ============================================================
# 4. Inspect unmatched IDs
# ============================================================

if len(adata_without_shape) > 0:

    print(
        f"AnnData cells without polygon: "
        f"{len(adata_without_shape):,}"
    )

    display(
        pd.DataFrame(
            {
                "instance_id":
                adata_without_shape[:20]
            }
        )
    )

else:

    print(
        "All AnnData cells have matching polygons."
    )


print()


if len(shape_without_adata) > 0:

    print(
        f"Polygons without AnnData cell: "
        f"{len(shape_without_adata):,}"
    )

    display(
        pd.DataFrame(
            {
                "instance_id":
                shape_without_adata[:20]
            }
        )
    )

else:

    print(
        "All polygons have matching AnnData cells."
    )

All AnnData cells have matching polygons.

All polygons have matching AnnData cells.


In [79]:
# ============================================================
# 5. Restrict table to cells with valid segmentation
# ============================================================

tab_adata.obs_names = (
    tab_adata.obs_names.astype(str)
)

tab_adata = tab_adata[
    matched_ids,
    :
].copy()


print(
    "AnnData after polygon matching:"
)

print(
    f"{tab_adata.n_obs:,} cells × "
    f"{tab_adata.n_vars:,} features"
)

AnnData after polygon matching:
43,565 cells × 1,010 features


In [80]:
# ============================================================
# 6. Add SpatialData linkage metadata
# ============================================================

tab_adata.obs[INSTANCE_KEY] = (
    tab_adata.obs_names.astype(str)
)


tab_adata.obs[REGION_KEY] = pd.Categorical(
    [REGION_NAME] * tab_adata.n_obs,
    categories=[REGION_NAME],
)


display(
    tab_adata.obs[
        [
            REGION_KEY,
            INSTANCE_KEY,
        ]
    ].head()
)

,region,instance_id
unique_cell_id,,
fov_1_cell_1,cell_boundaries,fov_1_cell_1
fov_1_cell_2,cell_boundaries,fov_1_cell_2
fov_1_cell_3,cell_boundaries,fov_1_cell_3
fov_1_cell_4,cell_boundaries,fov_1_cell_4
fov_1_cell_5,cell_boundaries,fov_1_cell_5


In [81]:
# ============================================================
# 7. Parse as SpatialData TableModel
# ============================================================

table = TableModel.parse(
    tab_adata,
    region=REGION_NAME,
    region_key=REGION_KEY,
    instance_key=INSTANCE_KEY,
)


print("=" * 70)
print("SPATIALDATA TABLE CREATED")
print("=" * 70)

print(
    f"Table shape: "
    f"{table.n_obs:,} cells × "
    f"{table.n_vars:,} features"
)

SPATIALDATA TABLE CREATED
Table shape: 43,565 cells × 1,010 features


In [82]:
# ============================================================
# 8. Inspect SpatialData table linkage metadata
# ============================================================

spatial_attrs = table.uns.get(
    "spatialdata_attrs"
)

print(
    "table.uns['spatialdata_attrs']:"
)

display(
    spatial_attrs
)

table.uns['spatialdata_attrs']:


{'region': 'cell_boundaries',
 'region_key': 'region',
 'instance_key': 'instance_id'}

In [83]:
# ============================================================
# 9. Validate Table → Shapes mapping
# ============================================================

table_instance_ids = (
    table.obs[INSTANCE_KEY]
    .astype(str)
)

shape_instance_ids = pd.Index(
    shapes.index.astype(str)
)


table_to_shape = (
    table_instance_ids
    .isin(shape_instance_ids)
)


print(
    "Table → polygon mapping:"
)

display(
    table_to_shape
    .value_counts()
    .rename_axis("matched")
    .reset_index(name="n_cells")
)

Table → polygon mapping:


,matched,n_cells
0,True,43565


In [84]:
# ============================================================
# 10. Validate Shapes → Table mapping
# ============================================================

table_id_index = pd.Index(
    table_instance_ids
)

shape_to_table = (
    shape_instance_ids
    .isin(table_id_index)
)


shape_to_table_summary = (
    pd.Series(
        shape_to_table,
        name="matched",
    )
    .value_counts()
    .rename_axis("matched")
    .reset_index(name="n_shapes")
)


print(
    "Polygon → table mapping:"
)

display(
    shape_to_table_summary
)

Polygon → table mapping:


,matched,n_shapes
0,True,43565


In [85]:
# ============================================================
# 11. Hard validation
# ============================================================

assert table.n_obs == len(shapes), (
    "Table and shape counts differ:\n"
    f"table={table.n_obs:,}\n"
    f"shapes={len(shapes):,}"
)


assert table_to_shape.all(), (
    "Some table cells do not have "
    "matching segmentation polygons."
)


assert shape_to_table.all(), (
    "Some segmentation polygons do not have "
    "matching table cells."
)


assert (
    table.obs[REGION_KEY]
    .astype(str)
    .eq(REGION_NAME)
    .all()
), (
    "Unexpected region values detected."
)


print(
    "All table ↔ segmentation "
    "validation checks passed."
)

All table ↔ segmentation validation checks passed.


In [86]:
# ============================================================
# 12. Expression / layer sanity checks
# ============================================================

print(
    "X shape:",
    table.X.shape,
)

print(
    "counts layer:",
    "counts" in table.layers,
)

print(
    "spatial coordinates:",
    "spatial" in table.obsm,
)


if "counts" in table.layers:

    print(
        "counts shape:",
        table.layers["counts"].shape,
    )


if "spatial" in table.obsm:

    print(
        "spatial shape:",
        table.obsm["spatial"].shape,
    )

X shape: (43565, 1010)
counts layer: True
spatial coordinates: True
counts shape: (43565, 1010)
spatial shape: (43565, 2)


In [87]:
# ============================================================
# 13. Final table-construction summary
# ============================================================

final_summary = pd.DataFrame(
    {
        "metric": [
            "Raw AnnData cells",
            "Valid segmentation polygons",
            "Matched table cells",
            "AnnData cells dropped",
            "Polygons without AnnData",
            "Features",
            "Table→Shape matched",
            "Shape→Table matched",
        ],
        "value": [
            adata_raw.n_obs,
            len(shapes),
            table.n_obs,
            len(adata_without_shape),
            len(shape_without_adata),
            table.n_vars,
            int(table_to_shape.sum()),
            int(shape_to_table.sum()),
        ],
    }
)

display(final_summary)

,metric,value
0,Raw AnnData cells,43565
1,Valid segmentation polygons,43565
2,Matched table cells,43565
3,AnnData cells dropped,0
4,Polygons without AnnData,0
5,Features,1010
6,Table→Shape matched,43565
7,Shape→Table matched,43565


In [88]:
# Transcript points. Explicit dtypes avoid the CellComp inference bug.
preview = pd.read_csv(
    TX,
    nrows=100,
)

print("Transcript file:")
print(TX)

print("\nPreview shape:")
print(preview.shape)

print("\nColumns:")
display(
    pd.DataFrame(
        {
            "column": preview.columns,
            "preview_dtype":
            preview.dtypes.astype(str).values,
        }
    )
)

display(
    preview.head()
)

Transcript file:
/research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_tx_file.csv.gz

Preview shape:
(100, 10)

Columns:


,column,preview_dtype
0,Unnamed: 0,int64
1,fov,int64
2,cell_ID,int64
3,x_global_px,float64
4,y_global_px,float64
5,x_local_px,float64
6,y_local_px,float64
7,z,int64
8,target,object
9,CellComp,object


,Unnamed: 0,fov,cell_ID,x_global_px,y_global_px,x_local_px,y_local_px,z,target,CellComp
0,1,1,0,-495086.706667,11128.573333,2904.96,3845.240,0,ITM2B,NaN
1,2,1,0,-495061.826667,11124.253333,2929.84,3840.920,0,FFAR2,NaN
2,3,1,0,-496281.546667,11010.733333,1710.12,3727.400,0,CHEK1,NaN
3,4,1,0,-495572.316667,10971.058333,2419.35,3687.725,0,COL4A1,NaN
4,5,1,0,-496020.986667,10958.453333,1970.68,3675.120,0,FFAR4,NaN


In [89]:
required_columns = [
    "x_global_px",
    "y_global_px",
    "target",
]

missing_columns = [
    c
    for c in required_columns
    if c not in preview.columns
]


if missing_columns:

    raise ValueError(
        "Transcript file is missing "
        "required columns:\n"
        f"{missing_columns}"
    )


print(
    "Required transcript columns present."
)

Required transcript columns present.


In [90]:
dtype_map = {
    "target": "object",
}

if "CellComp" in preview.columns:
    dtype_map["CellComp"] = "object"


print("Explicit dtype overrides:")
display(dtype_map)

Explicit dtype overrides:


{'target': 'object', 'CellComp': 'object'}

In [91]:
tx = dd.read_csv(
    TX,
    blocksize=None,
    assume_missing=True,
    dtype=dtype_map,
)


# Remove index-like export artifact
if "Unnamed: 0" in tx.columns:

    tx = tx.drop(
        columns=["Unnamed: 0"]
    )


print("Dask transcript columns:")
print(list(tx.columns))

print("\nDask dtypes:")
display(
    tx.dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

Dask transcript columns:
['fov', 'cell_ID', 'x_global_px', 'y_global_px', 'x_local_px', 'y_local_px', 'z', 'target', 'CellComp']

Dask dtypes:


,dtype
fov,float64
cell_ID,float64
x_global_px,float64
y_global_px,float64
x_local_px,float64
y_local_px,float64
z,float64
target,string
CellComp,string


In [92]:
tx_head = tx.head(
    10
)

display(tx_head)

,fov,cell_ID,x_global_px,y_global_px,x_local_px,y_local_px,z,target,CellComp
0,1.0,0.0,-495086.706667,11128.573333,2904.96,3845.240,0.0,ITM2B,<NA>
1,1.0,0.0,-495061.826667,11124.253333,2929.84,3840.920,0.0,FFAR2,<NA>
2,1.0,0.0,-496281.546667,11010.733333,1710.12,3727.400,0.0,CHEK1,<NA>
3,1.0,0.0,-495572.316667,10971.058333,2419.35,3687.725,0.0,COL4A1,<NA>
4,1.0,0.0,-496020.986667,10958.453333,1970.68,3675.120,0.0,FFAR4,<NA>
5,1.0,0.0,-495845.666667,10783.183333,2146.00,3499.850,0.0,MXRA8,<NA>
6,1.0,0.0,-494354.566667,10711.558333,3637.10,3428.225,0.0,S100A10,<NA>
7,1.0,0.0,-495960.316667,10661.483333,2031.35,3378.150,0.0,ENO1,<NA>
8,1.0,0.0,-497948.546667,10562.753333,43.12,3279.420,0.0,TPT1,<NA>
9,1.0,0.0,-495386.666667,10531.133333,2605.00,3247.800,0.0,TPT1,<NA>


In [93]:
print(
    "Example transcript targets:"
)

display(
    tx["target"]
    .dropna()
    .head(20)
)

Example transcript targets:


0       ITM2B
1       FFAR2
2       CHEK1
3      COL4A1
4       FFAR4
5       MXRA8
6     S100A10
7        ENO1
8        TPT1
9        TPT1
10     ADGRG3
11      FABP4
12       H4C3
13      ITM2A
14       TPT1
15      CCL17
16     COL6A1
17       ARF1
18      RPL37
19       NCR1
Name: target, dtype: string

In [94]:
# 如果想知道 transcript 数量：
# 但是，建议默认注释掉，因为 whole-slide transcript table 可能比较大。
# This triggers a full computation.
# Run only when you actually need the exact number.

# n_transcripts = tx.shape[0].compute()
# print(f"Total transcripts: {n_transcripts:,}")

In [95]:
points = PointsModel.parse(
    tx,
    coordinates={
        "x": "x_global_px",
        "y": "y_global_px",
    },
    feature_key="target",
    transformations={
        "global": Identity(),
    },
)


print(
    "SpatialData transcript Points created."
)

print(
    "Coordinate system: global"
)

print(
    "Feature key: target"
)

INFO     Column `z` in `data` will be ignored since the data is 2D.                                                
WARNING  The `feature_key` column target is categorical with unknown categories. Please ensure the categories are  
         known before calling `PointsModel.parse()` to avoid significant performance implications due to the need  
         for dask to compute the categories. If you did not use PointsModel.parse() explicitly in your code (e.g.  
         this message is coming from a reader in `spatialdata_io`), please report this finding.                    
SpatialData transcript Points created.
Coordinate system: global
Feature key: target


In [96]:
# ============================================================
# Assemble SpatialData
# ============================================================

S = SpatialData(
    points={
        "transcripts": points,
    },
    shapes={
        "cell_boundaries": shapes_model,
    },
    tables={
        "table": table,
    },
)

print(
    "SpatialData object constructed successfully."
)

SpatialData object constructed successfully.


In [97]:
# ============================================================
# Safe SpatialData inspection
#
# Avoid print(S) for the whole-slide object with the current
# SpatialData/Dask version combination.
# ============================================================

def inspect_sdata(sdata):

    print("=" * 70)
    print("SPATIALDATA STRUCTURE")
    print("=" * 70)

    print(
        "Points :",
        list(sdata.points.keys()),
    )

    print(
        "Shapes :",
        list(sdata.shapes.keys()),
    )

    print(
        "Tables :",
        list(sdata.tables.keys()),
    )

    print(
        "Images :",
        list(sdata.images.keys()),
    )

    print(
        "Labels :",
        list(sdata.labels.keys()),
    )

    print()

    print(
        "Coordinate systems:"
    )

    print(
        sdata.coordinate_systems
    )

    print()

    for name, t in sdata.tables.items():

        print(
            f"Table {name!r}: "
            f"{t.n_obs:,} cells × "
            f"{t.n_vars:,} features"
        )

    for name, sh in sdata.shapes.items():

        print(
            f"Shapes {name!r}: "
            f"{len(sh):,} polygons"
        )

    for name, pt in sdata.points.items():

        print(
            f"Points {name!r}:"
        )

        print(
            f"  columns = "
            f"{list(pt.columns)}"
        )


inspect_sdata(S)

SPATIALDATA STRUCTURE
Points : ['transcripts']
Shapes : ['cell_boundaries']
Tables : ['table']
Images : []
Labels : []

Coordinate systems:
['global']

Table 'table': 43,565 cells × 1,010 features
Shapes 'cell_boundaries': 43,565 polygons
Points 'transcripts':
  columns = ['x', 'y', 'target', 'y_local_px', 'fov', 'cell_ID', 'x_local_px', 'CellComp']


In [98]:
# ============================================================
# Cross-element validation
# ============================================================

TABLE_KEY = "table"
SHAPES_KEY = "cell_boundaries"
POINTS_KEY = "transcripts"

REGION_KEY = "region"
INSTANCE_KEY = "instance_id"


# ------------------------------------------------------------
# Required elements
# ------------------------------------------------------------

assert TABLE_KEY in S.tables
assert SHAPES_KEY in S.shapes
assert POINTS_KEY in S.points


s_table = S.tables[TABLE_KEY]
s_shapes = S.shapes[SHAPES_KEY]
s_points = S.points[POINTS_KEY]


# ------------------------------------------------------------
# Table ↔ Shapes counts
# ------------------------------------------------------------

print(
    "Table cells:",
    f"{s_table.n_obs:,}",
)

print(
    "Shape polygons:",
    f"{len(s_shapes):,}",
)


# ------------------------------------------------------------
# Table → Shapes
# ------------------------------------------------------------

table_ids = (
    s_table.obs[INSTANCE_KEY]
    .astype(str)
)

shape_ids = pd.Index(
    s_shapes.index.astype(str)
)

table_to_shape = (
    table_ids.isin(shape_ids)
)


# ------------------------------------------------------------
# Shapes → Table
# ------------------------------------------------------------

shape_to_table = (
    shape_ids.isin(
        pd.Index(table_ids)
    )
)


print()

print(
    "Table → Shape matched:",
    f"{table_to_shape.sum():,} / "
    f"{len(table_to_shape):,}",
)

print(
    "Shape → Table matched:",
    f"{shape_to_table.sum():,} / "
    f"{len(shape_to_table):,}",
)


# ------------------------------------------------------------
# Region
# ------------------------------------------------------------

region_values = (
    s_table.obs[REGION_KEY]
    .astype(str)
    .unique()
)

print()

print(
    "Table regions:",
    list(region_values),
)


# ------------------------------------------------------------
# Transcript schema
# ------------------------------------------------------------

required_tx_columns = {
    "x",
    "y",
    "target",
}

tx_columns = set(
    s_points.columns
)

print()

print(
    "Transcript columns:"
)

print(
    sorted(tx_columns)
)


# ------------------------------------------------------------
# Hard checks
# ------------------------------------------------------------

assert table_to_shape.all(), (
    "Some table cells do not map "
    "to segmentation polygons."
)

assert shape_to_table.all(), (
    "Some segmentation polygons do not map "
    "to table cells."
)

assert set(region_values) == {
    SHAPES_KEY
}, (
    "Unexpected table region mapping."
)

assert required_tx_columns.issubset(
    tx_columns
), (
    "Transcript Points are missing "
    "required x/y/target columns."
)


print()
print(
    "All cross-element validation checks passed."
)

Table cells: 43,565
Shape polygons: 43,565

Table → Shape matched: 43,565 / 43,565
Shape → Table matched: 43,565 / 43,565

Table regions: ['cell_boundaries']

Transcript columns:
['CellComp', 'cell_ID', 'fov', 'target', 'x', 'x_local_px', 'y', 'y_local_px']

All cross-element validation checks passed.


In [99]:
# ============================================================
# SpatialData output configuration
# ============================================================

WRITE_SDATA = False
# WRITE_SDATA = True

SDATA_TAG = "interactive"


SDATA_OUTPUT_DIR = (
    SPATIAL_DIR
    / "variants"
    / SDATA_TAG
)

SDATA_PATH = (
    SDATA_OUTPUT_DIR
    / f"GSM9046088_CosMx_{SDATA_TAG}.zarr"
)


print("=" * 70)
print("SPATIALDATA OUTPUT CONFIGURATION")
print("=" * 70)

print(
    "WRITE_SDATA:"
)

print(
    WRITE_SDATA
)

print(
    "\nSDATA_TAG:"
)

print(
    SDATA_TAG
)

print(
    "\nOutput directory:"
)

print(
    SDATA_OUTPUT_DIR
)

print(
    "\nOutput name:"
)

print(
    SDATA_PATH.name
)

print(
    "\nFull output path:"
)

print(
    SDATA_PATH
)

print(
    "\nCurrently exists:"
)

print(
    SDATA_PATH.exists()
)

SPATIALDATA OUTPUT CONFIGURATION
WRITE_SDATA:
False

SDATA_TAG:
interactive

Output directory:
/research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/spatial/variants/interactive

Output name:
GSM9046088_CosMx_interactive.zarr

Full output path:
/research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/spatial/variants/interactive/GSM9046088_CosMx_interactive.zarr

Currently exists:
True


In [100]:
# ============================================================
# Save SpatialData only when explicitly requested
# ============================================================

if WRITE_SDATA:

    SDATA_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    if SDATA_PATH.exists():

        raise FileExistsError(
            "Output SpatialData already exists:\n"
            f"{SDATA_PATH}\n\n"
            "Choose a different SDATA_TAG, or "
            "explicitly remove/replace the existing variant."
        )

    S.write(
        SDATA_PATH,
    )

    print(
        "SpatialData saved successfully:"
    )

    print(
        SDATA_PATH
    )

else:

    print(
        "WRITE_SDATA=False — "
        "SpatialData remains in memory only."
    )

WRITE_SDATA=False — SpatialData remains in memory only.


In [101]:
# ============================================================
# Verify saved SpatialData
# ============================================================

print(
    "Output exists:",
    SDATA_PATH.exists(),
)


if SDATA_PATH.exists():

    import spatialdata as sd

    S_test = sd.read_zarr(
        SDATA_PATH
    )

    print(
        "\nReload successful."
    )

    # Do NOT print(S_test) because of the
    # SpatialData 0.7.2 / Dask 2026.1.1 repr issue.

    inspect_sdata(
        S_test
    )

Output exists: True

Reload successful.
SPATIALDATA STRUCTURE
Points : ['transcripts']
Shapes : ['cell_boundaries']
Tables : ['table']
Images : []
Labels : []

Coordinate systems:
['global']

Table 'table': 43,565 cells × 1,010 features
Shapes 'cell_boundaries': 43,565 polygons
Points 'transcripts':
  columns = ['x', 'y', 'target', 'y_local_px', 'fov', 'cell_ID', 'x_local_px', 'CellComp']


In [102]:
# ============================================================
# Small transcript sanity check
# ============================================================

tx_sample = (
    S.points["transcripts"]
    .head(10)
)

display(
    tx_sample
)


print(
    "Targets in sample:"
)

print(
    tx_sample["target"]
    .astype(str)
    .tolist()
)

,x,y,target,y_local_px,fov,cell_ID,x_local_px,CellComp
0,-495086.706667,11128.573333,ITM2B,3845.240,1.0,0.0,2904.96,<NA>
1,-495061.826667,11124.253333,FFAR2,3840.920,1.0,0.0,2929.84,<NA>
2,-496281.546667,11010.733333,CHEK1,3727.400,1.0,0.0,1710.12,<NA>
3,-495572.316667,10971.058333,COL4A1,3687.725,1.0,0.0,2419.35,<NA>
4,-496020.986667,10958.453333,FFAR4,3675.120,1.0,0.0,1970.68,<NA>
5,-495845.666667,10783.183333,MXRA8,3499.850,1.0,0.0,2146.00,<NA>
6,-494354.566667,10711.558333,S100A10,3428.225,1.0,0.0,3637.10,<NA>
7,-495960.316667,10661.483333,ENO1,3378.150,1.0,0.0,2031.35,<NA>
8,-497948.546667,10562.753333,TPT1,3279.420,1.0,0.0,43.12,<NA>
9,-495386.666667,10531.133333,TPT1,3247.800,1.0,0.0,2605.00,<NA>


Targets in sample:
['ITM2B', 'FFAR2', 'CHEK1', 'COL4A1', 'FFAR4', 'MXRA8', 'S100A10', 'ENO1', 'TPT1', 'TPT1']
